# Set-Up

In [ ]:
# Imports for Generating & Viewing Data #

import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [ ]:
# Constants #

IMAGE_X = 320
IMAGE_Y = 320

GAUSSIAN_X = 48
GAUSSIAN_Y = 48

SCALE_FACTOR = 8

LABEL_X = IMAGE_X // SCALE_FACTOR
LABEL_Y = IMAGE_Y // SCALE_FACTOR

MIN_NUM_GAUSSIANS = 3
MAX_NUM_GAUSSIANS = 3

MIN_STD_X = 5
MAX_STD_X = 11

MIN_STD_Y = 5
MAX_STD_Y = 11

MIN_THETA = 0
MAX_THETA = np.pi

MIN_INTENSITY = 0.2
MAX_INTENSITY = 0.8

THRESHOLD = 0.9

SOFT_LABEL_SIGMA = 0.75

SEED = 0
rng = np.random.default_rng(SEED)

In [ ]:
# Utility Functions for Generating Data #

# TODO: consider param array
# TODO: look into parallelizing
def gen_data(num_images: int) -> tuple:

    img_arr = np.empty((num_images, IMAGE_Y, IMAGE_X, 1), dtype=np.float32)
    label_arr = np.empty((num_images, LABEL_Y, LABEL_X, 1), dtype=np.float32)
    param_arr = np.empty((num_images, MAX_NUM_GAUSSIANS, 6), dtype=np.float32)

    for i in tqdm(range(num_images)):
        img, label, param = img_gen()
        img_arr[i] = img.astype(np.float32) / 256.0  # Normalize values to [0, 1)
        label_arr[i] = label.astype(np.float32) / 256.0  # Normalize values to [0, 1)
        param_arr[i] = param

    print(f"[Images Shape]: {img_arr.shape}")
    print(f"[Labels Shape]: {label_arr.shape}")
    print(f"[Params Shape]: {param_arr.shape}")

    return (img_arr, label_arr, param_arr)


def img_gen() -> tuple:

    img = np.zeros(shape=(IMAGE_Y, IMAGE_X, 1))
    params = []

    num_gaussians = rng.integers(low=MIN_NUM_GAUSSIANS, high=MAX_NUM_GAUSSIANS + 1)
    for _ in range(num_gaussians):
        # Randomize params
        center_x = rng.integers(low=0 + GAUSSIAN_X // 2, high=IMAGE_X - GAUSSIAN_X // 2)
        center_y = rng.integers(low=0 + GAUSSIAN_Y // 2, high=IMAGE_Y - GAUSSIAN_Y // 2)

        std_x = rng.uniform(MIN_STD_X, MAX_STD_X)
        std_y = rng.uniform(MIN_STD_Y, MAX_STD_Y)
        theta = rng.uniform(MIN_THETA, MAX_THETA)

        intensity = MIN_INTENSITY + rng.random() * (MAX_INTENSITY - MIN_INTENSITY)

        # Generate Gaussian
        params.append((center_x, center_y, std_x, std_y, theta, intensity))
        gaussian = np.array(gaussian_gen(center_x, center_y, std_x, std_y, theta))

        # Add Gaussian to img
        img += gaussian * intensity

    # Apply gaussian labelling
    label = make_soft_label(params, LABEL_Y, LABEL_X, SCALE_FACTOR)

    # Convert to 8 bit int
    label = (label * 255).astype(np.uint8)
    img = (np.clip(img, 0.0, 1.0) * 255).astype(np.uint8)

    return (img, label, params)


X, Y = np.meshgrid(np.arange(IMAGE_X), np.arange(IMAGE_Y))


def gaussian_gen(
    center_x: float, center_y: float, std_x: float, std_y: float, theta: float, X=X, Y=Y
) -> np.ndarray:

    cos_theta_sqrd = np.power(np.cos(theta), 2)
    sin_theta_sqrd = np.power(np.sin(theta), 2)
    sin_cos_theta = np.sin(theta) * np.cos(theta)

    std_x_sqrd = np.power(std_x, 2)
    std_y_sqrd = np.power(std_y, 2)

    a = (cos_theta_sqrd) / (2 * std_x_sqrd) + (sin_theta_sqrd) / (2 * std_y_sqrd)
    b = -1 * (sin_cos_theta) / (2 * std_x_sqrd) + (sin_cos_theta) / (2 * std_y_sqrd)
    c = (sin_theta_sqrd) / (2 * std_x_sqrd) + (cos_theta_sqrd) / (2 * std_y_sqrd)

    gaussian = np.exp(
        -(
            a * (X - center_x) ** 2
            + 2 * b * (X - center_x) * (Y - center_y)
            + c * (Y - center_y) ** 2
        )
    )

    return np.expand_dims(gaussian, -1)


def make_soft_label(
    params, label_h, label_w, scale_factor, sigma=SOFT_LABEL_SIGMA
) -> np.ndarray:

    yy, xx = np.meshgrid(np.arange(label_h), np.arange(label_w), indexing="ij")
    label = np.zeros((label_h, label_w), dtype=np.float32)

    for center_x, center_y, std_x, std_y, theta, intensity in params:
        gx = center_x / scale_factor
        gy = center_y / scale_factor

        bump = np.exp(-((xx - gx) ** 2 + (yy - gy) ** 2) / (2 * sigma**2))
        label = np.maximum(label, bump)

    return label[..., None]

# Train / Load Qkeras Model

In [ ]:
# Imports for Training #

import tensorflow as tf
import keras.backend as K
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, Activation
from tensorflow.keras.activations import relu, sigmoid
from tensorflow.keras.models import Model
from qkeras import QConv2D, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

In [ ]:
# Create TF Datasets #
# TODO: look into using tf.data.Dataset.from_generator
TRAINING_DATASET_SIZE = 10000
VALIDATION_DATASET_SIZE = 1000
TEST_DATASET_SIZE = 1000
BATCH_SIZE = 50


train_img_arr, train_label_arr, _ = gen_data(TRAINING_DATASET_SIZE)
val_img_arr, val_label_arr, _ = gen_data(VALIDATION_DATASET_SIZE)
test_img_arr, test_label_arr, test_param_arr = gen_data(TEST_DATASET_SIZE)

train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_img_arr, train_label_arr))
    .shuffle(TRAINING_DATASET_SIZE, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_img_arr, val_label_arr))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# Qkeras Model Architecture #
TOTAL_BITS = 8
INTEGER_BITS = 0


w_quant = quantized_bits(
    TOTAL_BITS, INTEGER_BITS, symmetric=False, keep_negative=True, alpha=1
)
relu_quant = quantized_relu(TOTAL_BITS, INTEGER_BITS)

input_layer = Input(shape=(IMAGE_Y, IMAGE_X, 1))

x = QConv2D(
    filters=4,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_1_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(input_layer)
# x = BatchNormalization(name="b_1_1")(x)
x = QActivation(relu_quant, name="qact_1_1")(x)
# y = QConv2D(
#     filters=4,
#     kernel_size=3,
#     padding="same",
#     use_bias=False,
#     name="qconv2d_1_2",
#     kernel_quantizer=w_quant,
#     kernel_initializer="lecun_uniform",
# )(x)
# y = BatchNormalization(name="b_1_2")(y)
# x = Add(name="add_1")([x, y])
# x = QActivation(relu_quant, name="qact_1_2")(x)

x = QConv2D(
    filters=6,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_2_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
# x = BatchNormalization(name="b_2_1")(x)
x = QActivation(relu_quant, name="qact_2_1")(x)
# y = QConv2D(
#     filters=6,
#     kernel_size=3,
#     padding="same",
#     use_bias=False,
#     name="qconv2d_2_2",
#     kernel_quantizer=w_quant,
#     kernel_initializer="lecun_uniform",
# )(x)
# y = BatchNormalization(name="b_2_2")(y)
# x = Add(name="add_2")([x, y])
# x = QActivation(relu_quant, name="qact_2_2")(x)

x = QConv2D(
    filters=8,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_3_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
# x = BatchNormalization(name="b_3_1")(x)
x = QActivation(relu_quant, name="qact_3_1")(x)
# y = QConv2D(
#     filters=8,
#     kernel_size=3,
#     padding="same",
#     use_bias=False,
#     name="qconv2d_3_2",
#     kernel_quantizer=w_quant,
#     kernel_initializer="lecun_uniform",
# )(x)
# y = BatchNormalization(name="b_3_2")(y)
# x = Add(name="add_3")([x, y])
# x = QActivation(relu_quant, name="qact_3_2")(x)

x = Conv2D(8, 1, padding="same", use_bias=True, name="qconv2d_4")(x)
x = Activation(relu, name="qact_4")(x)

x = Conv2D(1, 1, padding="same", use_bias=True, name="qconv2d_5")(x)
x_prob = Activation(sigmoid, name="sigmoid_out")(x)

baby_yolo_qk = Model(inputs=input_layer, outputs=x_prob, name="baby_yolo")

In [ ]:
# Tensorflow Functions #
POS_WEIGHT = 5.0
NEG_WEIGHT = 1.0


def loss_wbce(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)

    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    weights = y_true * POS_WEIGHT + (1.0 - y_true) * NEG_WEIGHT

    return tf.reduce_mean(weights * bce)

In [ ]:
# Train or Load #
# TODO: Fix Load
TRAIN_MODEL = True
LOAD_MODEL = False

MODEL_NAME = ""

NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4


if TRAIN_MODEL and LOAD_MODEL:
    print("Are you sure you want to Train & Load ?")

elif TRAIN_MODEL:
    adam_optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        global_clipnorm=1.0,
    )

    baby_yolo_qk.compile(
        optimizer=adam_optimizer,
        loss=loss_wbce,
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=MIN_DELTA,
        restore_best_weights=True,
        verbose=1,
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1,
    )

    history = baby_yolo_qk.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=NUM_EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=1,
    )

elif LOAD_MODEL:
    pass

baby_yolo_qk.summary()

In [ ]:
# Save Model #
SAVE_MODEL = False

if SAVE_MODEL:
    baby_yolo_qk.save(f"Models/{MODEL_NAME}")

# Test QKeras Model

In [ ]:
# Utility Functions for Viewing Data #
# TODO: add comparison functions

In [ ]:
# Predict on Test Set #

predictions_qk = baby_yolo_qk.predict(test_img_arr)

print(f"[Test Shape]: {test_label_arr.shape}")
print(f"[Prediction Shape]: {predictions_qk.shape}")

# Convert to HLS

In [ ]:
# Imports for HLS #

import hls4ml
from hls4ml.utils import config_from_keras_model
from pprint import pprint

import os

xilinx_vitis = "/home/tools/Xilinx/2025/2025.2/Vitis"
xilinx_viv = "/home/tools/Xilinx/2025/2025.2/Vivado"
xilinx_hls = "/home/tools/Xilinx/2025/2025.2/Vitis"

# prepend Vitis bin directory to PATH
# os.environ["PATH"] = f"{xilinx_vitis}/bin:" + os.environ["PATH"]

os.environ["XILINX_HLS"] = xilinx_hls
os.environ["XILINX_VITIS"] = xilinx_vitis
os.environ["XILINX_VIVADO"] = xilinx_viv

In [ ]:
# Utils for HLS #

data = test_img_arr[:3].reshape(3, -1)
np.savetxt("Utils/Generated/tb_input_features.dat", data, fmt="%.6f")

data = test_label_arr[:3].reshape(3, -1)
np.savetxt("Utils/Generated/tb_output_predictions.dat", data, fmt="%.6f")

In [ ]:
# HLS4ML Config #
# TODO: look into pipeline
# TODO: verify config quantization

config = config_from_keras_model(baby_yolo_qk, granularity="name", backend="Vitis")

# Fifo Depth Optimization (greatly reduces BRAM)
# config["Flows"] = ["vitis:fifo_depth_optimization"]
# hls4ml.model.optimizer.get_optimizer("vitis:fifo_depth_optimization").configure(
#     profiling_fifo_depth=1000
# )

# Strategy
config["Model"]["Strategy"] = "Latency"
# config['Model']['PipelineStyle'] = 'pipeline'

# Reuse Factor
# config["Model"]["ReuseFactor"] = 4
# for layer_cfg in config["LayerName"].values():
#     if "ReuseFactor" in layer_cfg:
#         layer_cfg["ReuseFactor"] = 4

pprint(config)

In [ ]:
# Compile #
# Make sure you copy over the TB Data found in Utils for FOLO Depth Opt.

hls_model = hls4ml.converters.convert_from_keras_model(
    baby_yolo_qk,
    hls_config=config,
    output_dir="folo_hls4ml_Q_Gen",
    io_type="io_stream",  # io_parallel
    clock_period=2.0,
    clock_uncertainty="12.5%",
    backend="Vitis",
    part="xcku035-fbva676-2-e",
    project_name="folo",
)

hls_model.compile()

In [ ]:
# Predict #

predictions_hls4ml = hls_model.predict(test_img_arr).reshape(
    TEST_DATASET_SIZE, LABEL_Y, LABEL_X
)

print(f"[Test Shape]: {test_label_arr.shape}")
print(f"[Prediction Shape]: {predictions_hls4ml.shape}")

In [ ]:
# Visually Compare Predictions #
index = np.random.randint(low=0, high=TEST_DATASET_SIZE)

fig, axes = plt.subplots(1, 3)
im0 = axes[0].imshow(test_label_arr[index], cmap="viridis", interpolation="none")
axes[0].set_title("Test Label")
axes[0].axis("off")

im1 = axes[1].imshow(predictions_qk[index], cmap="viridis", interpolation="none")
axes[1].set_title("QK Prediction")
axes[1].axis("off")

im2 = axes[2].imshow(predictions_hls4ml[index], cmap="viridis", interpolation="none")
axes[2].set_title("HLS Prediction")
axes[2].axis("off")

plt.show()

In [ ]:
# Build Model #
# vitis_hls build_prj.tcl
# vitis-run --mode hls --tcl build_prj.tcl

# hls_model.build()